In [1]:
import json
import os
import requests, time


def load_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text
    
# file_path = "7-finetuning-data/alpaca_data.json"
file_path = "7-finetuning-data/instruction-data.json"

data = load_file(file_path)
print("Number of entries:", len(data))

Number of entries: 1100


# Dataset prep

In [2]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [3]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # Add an <|endoftext|> token
        new_item += [pad_token_id]
        # Pad sequences to max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # New: Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # New: Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst)
    targets_tensor = torch.stack(targets_lst)

    return inputs_tensor, targets_tensor

inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    allowed_max_length=1024
)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


### DataLoaders

In [4]:
from torch.utils.data import DataLoader
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [5]:
from llms_from_scratch_utils import GPT_355M_FILE, GPT_MODEL_CONFIGS, BASE_GPT_CONFIG, PATH_PREFIX
from llms_rasbt_repo import GPTModel
import torch
BASE_GPT_CONFIG.update(GPT_MODEL_CONFIGS["gpt2-medium (355M)"])

gpt = GPTModel(BASE_GPT_CONFIG)
gpt.load_state_dict(torch.load(GPT_355M_FILE, weights_only=True))
gpt.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [6]:
from llms_from_scratch_utils import generate, generate_and_print_sample
import tiktoken
# print(generate(gpt, "Every effort moves you", max_new_tokens=30, context_size=BASE_GPT_CONFIG["context_length"], top_k=25, temperature=1.4))

tokenizer = tiktoken.get_encoding("gpt2")
# generate_and_print_sample(model=gpt, start_context="Every effort moves you", tokenizer=tokenizer)


## LoRA layer

In [7]:
import math
class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha
        self.rank = rank
    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [8]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)
    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [9]:
def replace_linear_with_lora(model, rank, alpha):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            # replace linear with lora
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            # recursively apply to child modules
            replace_linear_with_lora(module, rank, alpha)

In [10]:
from copy import deepcopy
gpt_lora = deepcopy(gpt)


total_params = sum(p.numel() for p in gpt_lora.parameters() if p.requires_grad)
print(f"{total_params:,}")
for param in gpt_lora.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in gpt_lora.parameters() if p.requires_grad)
print(f"{total_params:,}")


replace_linear_with_lora(gpt_lora, rank=16, alpha=16)
total_params = sum(p.numel() for p in gpt_lora.parameters() if p.requires_grad)
print(f"{total_params:,}")



406,286,336
0
7,898,384


# Train LoRA model

In [15]:
import time
from llms_from_scratch_utils import train_model_simple

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(gpt.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 5

# train_losses, val_losses, tokens_seen = train_model_simple(
#     gpt, train_loader, val_loader, optimizer, device="cpu",
#     num_epochs=num_epochs, eval_freq=5, eval_iter=5,
#     start_context=format_input(val_data[0]), tokenizer=tokenizer
# )

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

# name sft file with timestamp and use the base model name & path prefix
lora_model = f"gpt2-medium-355M-lora"
lora_sft_file_name = f"{PATH_PREFIX}{lora_model}.pth"

# torch.save(gpt_lora.state_dict(), sft_file_name)
# print(f"Model saved as {lora_sft_file_name}")

Training completed in 0.00 minutes.


In [12]:
gpt_lora.load_state_dict(torch.load(lora_sft_file_name, weights_only=True))
gpt_lora.eval()


input_txt = format_input(val_data[20])
generated_text  = generate(
    model=gpt_lora,
    start_context=input_txt,
    max_new_tokens=50,
    context_size=1024,
    eos_id=50256,
)

response_text = (
    generated_text[len(input_txt):]
    .replace("### Response:", "")
    .strip()
)
print(val_data[20])
print('----')
print(generated_text)
print('----')

print(response_text)

{'instruction': 'Reverse the order of the given phrase.', 'input': 'sun and moon', 'output': 'moon and sun'}
----
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Reverse the order of the given phrase.

### Input:
sun and moon

### Response:
The sun and moon are in the opposite order.
----
The sun and moon are in the opposite order.


# Full Model Finetune

In [13]:
import time
from llms_from_scratch_utils import train_model_simple

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(gpt.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 5

train_losses, val_losses, tokens_seen = train_model_simple(
    gpt, train_loader, val_loader, optimizer, device="cpu",
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

# name sft file with timestamp and use the base model name & path prefix
sft_model = f"gpt2-medium-355M-sft"
sft_file_name = f"{PATH_PREFIX}{sft_model}.pth"

torch.save(gpt.state_dict(), sft_file_name)
print(f"Model saved as {sft_file_name}")

Below is an instruction that describes a task. Write a response that appropriately completes the request.  ### Instruction: Convert the active sentence to passive: 'The chef cooks the meal every day.'  ### Response:  The chef cooks the meal every day.  ### Instruction:  Convert the active sentence to passive: 'The chef cooks the meal every day.'  ### Response:  The chef cooks the
Ep 1 (Step 000000): Train loss 2.637, Val loss 2.626
Ep 1 (Step 000005): Train loss 1.174, Val loss 1.102
Ep 1 (Step 000010): Train loss 0.872, Val loss 0.945
Ep 1 (Step 000015): Train loss 0.856, Val loss 0.906
Ep 1 (Step 000020): Train loss 0.776, Val loss 0.881
Ep 1 (Step 000025): Train loss 0.753, Val loss 0.859


KeyboardInterrupt: 

# Dump model output

In [16]:
from tqdm import tqdm

def generate_model_responses(data, model):
    for i, entry in tqdm(enumerate(data), total=len(data)):
        input_txt = format_input(entry)
        generated_text  = generate(
            model=model,
            start_context=input_txt,
            max_new_tokens=50,
            context_size=1024,
            eos_id=50256
            )
        response_text = (generated_text[len(input_txt):]
        .replace("### Response:", "")
        .strip())
        
        train_data[i]['model_response'] = response_text

generate_model_responses(train_data[:30], gpt_lora)
with open(f'{PATH_PREFIX}train_data_with_{lora_model}_responses.json', 'w') as f:
    json.dump(train_data, f, indent=4)

generate_model_responses(test_data, gpt_lora)
with open(f'{PATH_PREFIX}test_data_with_{lora_model}_responses.json', 'w') as f:
    json.dump(test_data, f, indent=4)

generate_model_responses(train_data[:30], gpt)
with open(f'{PATH_PREFIX}train_data_with_{sft_model}_responses.json', 'w') as f:
    json.dump(train_data, f, indent=4)

generate_model_responses(test_data, gpt)
with open(f'{PATH_PREFIX}test_data_with_{sft_model}_responses.json', 'w') as f:
    json.dump(test_data, f, indent=4)


100%|██████████| 30/30 [02:41<00:00,  5.38s/it]


NameError: name 'sft_model' is not defined

## Ollama


In [ ]:
import requests
from llms_from_scratch_utils import query_ollama

model="gpt-oss:20b",
# If you used OLLAMA_HOST=127.0.0.1:11435 ollama serve
# update the address from 11434 to 11435
url="http://localhost:11434/api/chat"    
# Send the POST request

data = {
    "model": "gpt-oss:20b",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello in one sentence."}
    ],
}
with requests.post(url, json=data, stream=True, timeout=30) as r:
    if not r.ok:
        print(r.status_code, r.text)
        r.raise_for_status()
    response_data = ""
    for line in r.iter_lines(decode_unicode=True):
        if not line:
            continue
        response_json = json.loads(line)
        if "message" in response_json:
            response_data += response_json["message"]["content"]

print(response_data)

Hello! I hope you're having a great day.


In [ ]:
query_ollama("Say hello in one sentence.")

'Hello!'

In [ ]:
# read test_data_with_gpt2_medium_355M_lora_responses.json
with open(f'{PATH_PREFIX}test_data_with_model_responses.json', 'r') as f:
    test_data = json.load(f)
from tqdm import tqdm
def generate_model_scores(json_data):
    scores = []
    # loop over json_data in tqdm
    for entry in tqdm(json_data, total=len(json_data), desc="Generating model scores"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`"
            f"score the model's response `{entry['model_response']}`"
            f"on a scale of 0 to 100, where 0 is the worst and 100 is the best score."
            f"Your output should be in the following format: Score: <integer> \nRationale: <explanation>"
        )
        judge_response = query_ollama(prompt)
        # extract oput the number from string "Score: 0 Rationale:  The instruction asks for the author of *Pride and Prejudice*, whose correct answer is **Jane Austen**. The model incorrectly identifies George Bernard Shaw as the author, which is a factual error. Because the response does not address the instruction correctly, it receives the lowest possible score on the 0‑to‑100 scale.in"
        # score_str = judge_response.split("Score: ")[1].split("Rationale:")[0].strip()
        try:
            score = int(judge_response.split("Score: ")[1].split("Rationale:")[0].strip())
            scores.append(score)
        except:
            print(f"Error parsing score for: {judge_response}")

    return scores

# scores = generate_model_scores(test_data)
scores = generate_model_scores(train_data[:30])
print(f"Number of scores: {len(scores)} out of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores)}")








Generating model scores: 100%|██████████| 30/30 [03:17<00:00,  6.58s/it]

Number of scores: 30 out of 110
Average score: 32.233333333333334
